In [1]:
!pip install rdkit-pypi
!pip install torch torch-geometric
!pip install pandas numpy scikit-learn
!pip install shap

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.7 MB/s eta 0:00:00


In [2]:
import pandas as pd

df = pd.read_csv("tox21.csv")
df.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [4]:
!pip install rdkit==2023.9.5 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 40.9 MB/s eta 0:00:00


In [6]:
from rdkit import Chem
import torch
from torch_geometric.data import Data

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([
            atom.GetAtomicNum(),
            atom.GetTotalDegree(),
            atom.GetFormalCharge(),
            atom.GetHybridization(),
            atom.GetTotalNumHs()
        ])

    edges = []
    for bond in mol.GetBonds():
        edges.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
        edges.append([bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()])

    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

In [8]:
df.columns

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles'],
      dtype='object')

In [9]:
TARGET = "SR-MMP"   # You can choose any one label from df.columns

data_list = []

for i in range(len(df)):
    smi = df.loc[i, "smiles"]      # molecule string
    y = df.loc[i, TARGET]          # toxicity label (0 or 1)

    g = smiles_to_graph(smi)       # convert SMILES to graph
    if g is not None:
        g.y = torch.tensor([float(y)], dtype=torch.float)
        data_list.append(g)

print("Total valid graphs created:", len(data_list))

[12:54:58] WARNING: not removing hydrogen atom without neighbors


Total valid graphs created: 7831


In [11]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool

class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(5, 32)
        self.conv2 = GCNConv(32, 64)
        self.lin = nn.Linear(64, 1)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return torch.sigmoid(self.lin(x))

In [15]:
df["SR-MMP"].unique()

array([ 0., nan,  1.])

In [16]:
df = df[df["SR-MMP"].isin([0, 1])]   # keep only valid labels
df = df.dropna(subset=["SR-MMP"])    # drop missing
df = df.reset_index(drop=True)       # reset index

In [17]:
TARGET = "SR-MMP"

data_list = []

for i in range(len(df)):
    smi = df.loc[i, "smiles"]
    y = df.loc[i, TARGET]

    g = smiles_to_graph(smi)
    if g is not None:
        g.y = torch.tensor([float(y)], dtype=torch.float)
        data_list.append(g)

len(data_list)

[13:02:25] WARNING: not removing hydrogen atom without neighbors


5810

In [18]:
from torch_geometric.loader import DataLoader
import torch.optim as optim

loader = DataLoader(data_list, batch_size=32, shuffle=True)

model = GNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    loss_sum = 0
    for batch in loader:
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, batch.batch)
        target = batch.y.view(-1, 1)

        # Ensure target is float and within 0–1
        target = target.clamp(0, 1)

        loss = criterion(out, target)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()

    print(f"Epoch {epoch+1}, Loss: {loss_sum:.4f}")

Epoch 1, Loss: 79.6528
Epoch 2, Loss: 77.9375
Epoch 3, Loss: 77.2423
Epoch 4, Loss: 76.7358
Epoch 5, Loss: 76.1608


In [19]:
torch.save(model.state_dict(), "toxigenx_model.pt")